<a href="https://colab.research.google.com/github/jinyu1124/andriod/blob/master/Colab-WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:01:52
🔁 Restarting kernel...


In [2]:
%%writefile /content/setup.sh
set -e

cd /content
if [ ! -d "GPT-SoVITS" ]; then
  git clone https://github.com/RVC-Boss/GPT-SoVITS.git
fi
cd GPT-SoVITS

# 1. 强制清理并创建干净的虚拟环境
if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    conda env remove -n GPTSoVITS -y
fi
conda create -n GPTSoVITS -c conda-forge python=3.10 -y

source activate GPTSoVITS
pip install ipykernel

# 2. 安装核心计算库 (GPU CUDA 12.4 版本)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. 安装依赖，并强制修复 Jinja2 导致网页崩溃的 Bug
pip install -r requirements.txt
pip install "jinja2<3.1.5" --force-reinstall

# 4. 安装音视频处理核心组件 FFmpeg
conda install -c conda-forge ffmpeg -y

# 5. 执行官方下载（若路径解析报错则跳过）
python tools/dlmodels.py || echo "[提示] 官方下载脚本触发 Bug，自动切换为手动模式..."

# 6. 手动强制拉取中文核心支持大模型
apt-get install git-lfs -y
git lfs install
mkdir -p GPT_SoVITS/pretrained_models
cd GPT_SoVITS/pretrained_models
rm -rf chinese-roberta-wwm-ext-large chinese-hubert-base
git clone https://huggingface.co/hfl/chinese-roberta-wwm-ext-large
git clone https://huggingface.co/TencentGameMate/chinese-hubert-base

Writing /content/setup.sh


In [ ]:
!cd /content && bash setup.sh

Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5914, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 5914 (delta 40), reused 14 (delta 14), pack-reused 5849 (from 4)
Receiving objects: 100% (5914/5914), 14.18 MiB | 11.70 MiB/s, done.
Resolving deltas: 100% (3369/3369), done.
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ | done

## Package Plan ##

  environment location: /usr/local/envs/GPTSoVITS

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         

In [ ]:
!cd /content/GPT-SoVITS && export is_share=True && /usr/local/envs/GPTSoVITS/bin/python webui.py